# Qwen SFT and two custom DPO runs
Enable T4 x2 and Internet. This notebook is a fresh reproduction workflow, not evidence of a new executed experiment. Use new run directories; save the completed version to preserve best/final checkpoints. The original report uses the already-saved checkpoints.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
ROOT=Path('/kaggle/working/cs5326-pa2-small-model-alignment')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/salehfarooq/cs5326-pa2-small-model-alignment.git',str(ROOT)],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH']=str(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import torch
before=torch.__version__
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-qwen.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.','--no-deps'],check=True)
from importlib.metadata import version
assert torch.__version__==before
assert torch.cuda.device_count()==2
print({name:version(name) for name in ['torch','transformers','datasets']})
print([torch.cuda.get_device_name(i) for i in range(2)])
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_public_adapter_smoke.py'],check=True)


In [ ]:
RUNS=Path('/kaggle/working/pa2_reproduction')
RUNS.mkdir(exist_ok=True)
for track in ['sft','hh','uf']:
    cmd=[sys.executable,'-u','-m','scripts.train_qwen','--track',track,
         '--data-dir',str(ROOT/'data/general'/track),
         '--output-dir',str(RUNS/track),'--full-run']
    if track!='sft':
        cmd+=['--sft-best',str(RUNS/'sft/best'),'--lr','1e-6','--beta','0.1',
              '--development-prompts',str(ROOT/'data/development/dpo_quality_40.json'),
              '--generation-updates','100,200,300,400,terminal',
              '--decoding-config',str(ROOT/'configs/training_diagnostic_decoding.json'),
              '--exclusion-bank',str(ROOT/'data/exclusions/alpaca_eval.jsonl'),
              '--exclusion-bank',str(ROOT/'eval_sets/hh_rlhf_eval_300_seed42.json')]
    subprocess.run(cmd,check=True)
    state=json.loads((RUNS/track/'run.json').read_text())
    assert state['status']=='completed'
    print(track,state['updates'],state['best_metrics'])
